# P10 · Valoración fundamental y macroeconomía
**QuantLab | Módulo 1 · Fundamentos financieros | Práctica 10/12**

## 1. Objetivo
Aprender a valorar activos con flujos de caja, interpretar tipos de interés e inflación y construir un estudio de eventos macroeconómicos sobre BTC/USD. El resultado es un prototipo de investigación, no una recomendación de inversión.

**Ruta:** valoración → indicadores → publicación y disponibilidad → reacción del mercado → validación → ejercicios.

Los datos sintéticos están identificados y sirven para comprobar el método. No son cotizaciones ni publicaciones reales. Los bloques opcionales permiten descargar series reales de FRED e importar eventos y precios propios. No se usan claves de trading ni se envían órdenes.

Las funciones están en **`src/quantlab/p10/valuation_macro.py`** y sus pruebas en **`tests/test_valuation_macro.py`** (`pytest` desde la raíz del proyecto). Los ejercicios están resueltos en la sección 8.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from pathlib import Path
from IPython.display import display
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p10.valuation_macro import (FRED_SERIES, SURPRISE_GROUPS, align_events, availability_report, bond_price,
                                          bond_shock, dcf, dcf_sensitivity, development_horizons, fetch_fred,
                                          fisher_comparison, fred_indicators, inverted_curve, macro_features, manual_pv,
                                          present_value, real_return, split_events, surprise_groups,
                                          synthetic_events_and_prices, synthetic_macro, validate_events_prices,
                                          value_per_share, yoy_by_date)

SEED = 42
DATA_MODE = 'synthetic'  # 'csv' para estudio de eventos propios
EVENTS_CSV = 'macro_events.csv'
PRICES_CSV = 'market_prices.csv'
RUN_FRED = False       # True para descargar series reales (descriptivas)
FRED_START = '2022-01-01'
FRED_END = '2025-12-31'
EXPORT = False
OUTPUT_DIR = Path('p10_output')
plt.rcParams.update({'figure.figsize': (10, 3.7), 'axes.grid': True,
                     'grid.alpha': .25, 'figure.dpi': 115})
pd.options.display.float_format = '{:,.4f}'.format

## 2. Valor temporal del dinero
Un euro hoy no equivale a un euro dentro de cinco años: podemos invertirlo, existe inflación y el cobro futuro puede ser incierto.

$$PV=\frac{CF_t}{(1+r)^t}$$

`r` es la tasa anual de descuento (decimal); `t` son años. El descuento aumenta cuando la tasa aumenta. En un bono sin riesgo de impago modelado, los cupones y el principal se descuentan al rendimiento exigido, suponiendo curva plana y pagos anuales.

In [ ]:
print(inspect.getsource(present_value))
print(inspect.getsource(bond_price))

rates = np.array([.02,.03,.04,.05,.06,.07])
bonds = pd.DataFrame({'yield_pct': rates*100,
                      'price_eur': [bond_price(1000,.04,r,5) for r in rates]})
display(bonds)
fig, ax = plt.subplots()
ax.plot(bonds.yield_pct,bonds.price_eur,marker='o',color='#31688e')
ax.axhline(1000,color='gray',ls='--',label='Nominal')
ax.set(title='Bono hipotético: nominal 1.000 €, cupón 4 %, 5 años',
       xlabel='Rendimiento exigido (%)',ylabel='Precio (€)')
ax.legend(); plt.tight_layout(); plt.show()

### Interpretación
Si el rendimiento exigido supera el cupón, el precio cae por debajo del nominal. No confundir el cupón fijo con la rentabilidad obtenida al comprar el bono a otro precio.

## 3. Descuento de flujos de caja (DCF)
Para una empresa hipotética, descontamos los flujos libres para la empresa (FCFF) al WACC. El valor terminal supone crecimiento perpetuo estable a una tasa `g` inferior a `r`.

$$EV=\sum_{t=1}^{N}\frac{FCFF_t}{(1+r)^t}+\frac{FCFF_N(1+g)}{(r-g)(1+r)^N}$$

Valor del capital = valor de empresa − deuda financiera neta (modelo simplificado). Un negocio puede ser excelente y resultar caro si su precio supera su valoración. Bitcoin no genera FCFF por poseerlo: este DCF **no** es un modelo de valoración directa de BTC.

In [ ]:
print(inspect.getsource(dcf))

flows = 100*1.04**np.arange(5)  # millones EUR; primer año 100
base = dcf(flows,.08,.02)
net_debt, shares, market_price = 300, 100, 15  # millones, millones, EUR/acción
intrinsic = value_per_share(flows,.08,.02,net_debt,shares)
print(f"EV: {base['enterprise_value']:.2f} M€ | Valor por acción: {intrinsic:.2f} €")
print(f"Valor terminal: {base['pv_terminal']:.2f} M€ ({base['pv_terminal']/base['enterprise_value']:.1%} del EV)")
print(f"Diferencia sobre precio observado: {(intrinsic/market_price-1)*100:.2f}%")
r_grid = [.06,.07,.08,.09,.10,.11,.12]
g_grid = [.00,.01,.02,.03,.04]
sensitivity = dcf_sensitivity(flows, r_grid, g_grid, net_debt, shares)
fig, ax = plt.subplots(figsize=(8,4.7))
im=ax.imshow(sensitivity, cmap='Blues',aspect='auto')
ax.set(xticks=range(len(g_grid)),xticklabels=sensitivity.columns,
       yticks=range(len(r_grid)),yticklabels=sensitivity.index,
       xlabel='Crecimiento perpetuo g',ylabel='Tasa de descuento r',
       title='Sensibilidad DCF: valor estimado por acción (€)')
for i in range(len(r_grid)):
    for j in range(len(g_grid)):
        v=sensitivity.iloc[i,j]; ax.text(j,i,f'{v:.1f}',ha='center',va='center',
              color='white' if im.norm(v)>.57 else '#172736')
fig.colorbar(im,ax=ax,label='€/acción'); plt.tight_layout(); plt.show()

### Interpretación
El DCF es sensible a dos supuestos que no observamos directamente: tasa de descuento y crecimiento futuro. Una diferencia positiva frente al precio de mercado no demuestra infravaloración sin una estimación fiable de esos supuestos.

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Bonos y DCF: el valor depende de supuestos que no se observan</b>

- **Bono (cupón 4 %, 5 años):** a un rendimiento del 4 % vale exactamente 1.000 € (a la par); al 2 %, 1.094,27 €; al 7 %, 876,99 €. Precio y rendimiento se mueven en sentido contrario.
- **DCF:** EV de 1.783,43 M€, de los que **1.353,52 M€ (el 75,9 %) son valor terminal**: tres cuartas partes del valor dependen de lo que pase después del año 5. Valor por acción: 14,83 €, un 1,1 % por debajo del precio de 15 €.
- **Sensibilidad:** con los mismos flujos, el valor por acción va de **6,40 €** (r = 12 %, g = 0 %) a **47,00 €** (r = 6 %, g = 4 %). Mover r o g un punto cambia más el valor que la diferencia con el precio de mercado. Con esa incertidumbre, un −1,1 % no dice nada sobre si la acción está cara o barata.

</div>

## 4. Inflación y curva de tipos
- Inflación interanual: $CPI_t/CPI_{t-12}-1$ sobre una serie mensual.
- Rentabilidad real exacta: $(1+r_{nominal})/(1+\pi)-1$, con ambos términos del mismo horizonte.
- Pendiente de curva: rendimiento a 10 años menos rendimiento a 2 años (en puntos porcentuales; se multiplica por 100 para bps).

La inversión de la curva puede reflejar expectativas de desaceleración, pero no demuestra causalidad ni garantiza una recesión.

> ⚠️ $CPI_{t-12}$ debe ser **el mismo mes del año anterior**, no la fila 12 posiciones atrás. Si falta un mes en la serie, `pct_change(12)` por posición compararía con el mes equivocado. `yoy_by_date` reindexa a una rejilla mensual completa antes de calcular (ver E7: en FRED falta octubre de 2025).

In [ ]:
print(inspect.getsource(yoy_by_date))
print(inspect.getsource(macro_features))

macro = synthetic_macro(SEED)
print(f'Rentabilidad real con nominal 5% e inflación 3%: {real_return(.05,.03):.2%}')
fig,ax=plt.subplots()
macro[['inflation_yoy_pct','policy_rate_pct','yield_2y_pct','yield_10y_pct']].plot(ax=ax)
ax.set(title='Indicadores macroeconómicos — sintéticos',xlabel='Mes de referencia',ylabel='%')
ax.legend(['IPC interanual','Tipo de política','Bono 2 años','Bono 10 años'],ncol=2)
plt.tight_layout(); plt.show()

## 5. Datos reales: FRED y ALFRED (opcional, descriptivo)
Fuente: Federal Reserve Bank of St. Louis, FRED. Series: [CPIAUCSL](https://fred.stlouisfed.org/series/CPIAUCSL), [FEDFUNDS](https://fred.stlouisfed.org/series/FEDFUNDS), [DGS2](https://fred.stlouisfed.org/series/DGS2), [DGS10](https://fred.stlouisfed.org/series/DGS10), [UNRATE](https://fred.stlouisfed.org/series/UNRATE), [CBBTCUSD](https://fred.stlouisfed.org/series/CBBTCUSD) (Coinbase, BTC/USD).

El siguiente bloque descarga CSV de la versión disponible hoy: es útil para explorar, pero **no** para simular qué sabíamos en una fecha antigua. No se rellenan fines de semana ni se mezclan frecuencias. El IPC necesita 12 meses anteriores para calcular inflación interanual; se respeta su frecuencia mensual. Para un backtest utiliza versiones históricas de [ALFRED](https://fred.stlouisfed.org/docs/api/fred/realtime_period.html) y horas de publicación verificadas por separado. ALFRED por sí solo no aporta el consenso ni necesariamente la hora intradía exacta.

In [ ]:
print(inspect.getsource(fetch_fred))

fred={}
if RUN_FRED:
    for series in FRED_SERIES:
        fred[series]=fetch_fred(series,FRED_START,FRED_END)  # si falla, lanza el error
    print('Descarga UTC:',pd.Timestamp.now(tz='UTC'))
    display(pd.DataFrame({'n':{k:len(v) for k,v in fred.items()},
                          'last':{k:v.index.max() for k,v in fred.items()}}))
    curve=(fred['DGS10']-fred['DGS2']).dropna()*100
    curve.plot(title='FRED · curva 10 años − 2 años (bps)',ylabel='bps'); plt.show()
else:
    print('FRED desactivado. Cambia RUN_FRED a True para consultar series reales.')

## 6. Estudio de eventos: sorpresa macroeconómica
Una inflación del 3% puede ser positiva o negativa para el mercado dependiendo de lo que se esperaba. Definimos sorpresa en puntos porcentuales:

$$s_t=actual_t-consenso_t$$

Un evento es observable a partir de `available_at`, no del mes de referencia. El consenso debe haberse registrado antes de publicar el dato. La reacción se mide desde el último precio conocido **estrictamente antes** del evento hasta el precio observado al horizonte fijado; esta medida incluye el movimiento de noticia y no es un retorno ejecutable tras conocerla.

**Contrato CSV** (modo `csv`): `macro_events.csv` contiene una fila por publicación de IPC interanual de EE. UU., columnas `event_id,reference_month,available_at,consensus_at,actual_pct,consensus_pct` (timestamps con zona; tasas en %). `market_prices.csv`: `timestamp,price`, BTC/USD en un mismo mercado, timestamp de disponibilidad de precio (si procede de vela, usar final de vela, nunca inicio). No se mezclan unidades, revisiones posteriores, activos ni tipos de evento. Para reproducibilidad guarda asimismo fuente, fecha de extracción y vintage en tu repositorio.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿los eventos y los precios son coherentes?</b>

`validate_events_prices` se detiene con un error si:

- En modo CSV, algún timestamp no tiene zona horaria.
- Hay `event_id` u horas de publicación duplicados, o falta la hora de publicación o del consenso.
- El consenso se registró **después** de la publicación (sería información futura).
- El dato o el consenso no son finitos, o hay precios duplicados, vacíos o ≤ 0.

Después ordena por hora y calcula `surprise_pp` = dato − consenso.

</div>

In [ ]:
print(inspect.getsource(synthetic_events_and_prices))
print(inspect.getsource(validate_events_prices))

if DATA_MODE=='synthetic':
    events,prices=synthetic_events_and_prices(SEED)
elif DATA_MODE=='csv':
    events=pd.read_csv(EVENTS_CSV); prices=pd.read_csv(PRICES_CSV)
else:
    raise ValueError('DATA_MODE debe ser synthetic o csv')
events, prices = validate_events_prices(events, prices, require_tz=DATA_MODE=='csv')
display(events.head(4)); print(f'{DATA_MODE}: {len(events)} eventos, {len(prices):,} precios')

### Alineación temporal
Usamos un `merge_asof` hacia atrás para que nunca se seleccione un precio posterior al instante consultado. Se exige una antigüedad máxima de 60 minutos; si no hay precio disponible, queda ausente. Los precios a horizontes posteriores son etiquetas de evaluación, no features. También marcaremos ventanas solapadas, que reducirían la independencia entre eventos.

In [ ]:
print(inspect.getsource(align_events))
print(inspect.getsource(split_events))

results = align_events(events,prices,(1,6,24),'60min')
# 65/35 cronológico. Purgar objetivos que cruzan la frontera, excluir solapes.
results, cutoff = split_events(results, .65, 24)
valid=results.loc[results.eligible].copy()
# Umbral pedagógico fijado antes de observar resultados; no optimizar en validación.
THRESHOLD_PP=.15
valid = surprise_groups(valid, THRESHOLD_PP)
summary=(valid.groupby(['split','surprise_group'])['return_1h_bps']
         .agg(n='count',mean_bps='mean',median_bps='median'))
print('Frontera:', cutoff, '| eventos elegibles:', int(results.eligible.sum()), 'de', len(results))
display(summary)

In [ ]:
fig,ax=plt.subplots()
for split,color in [('development','#31688e'),('validation','#d17a38')]:
    sample=valid[valid.split==split]
    ax.scatter(sample.surprise_pp,sample.return_1h_bps,label=split,color=color,alpha=.8)
ax.axhline(0,color='gray',lw=1); ax.axvline(0,color='gray',lw=1)
ax.set(title=f'Sorpresa de IPC y reacción de BTC a 1 h — {DATA_MODE}',
       xlabel='Sorpresa IPC (puntos porcentuales)',ylabel='Retorno desde precio previo (bps)')
ax.legend(); plt.tight_layout(); plt.show()

cols=[f'return_{h}h_bps' for h in (1,6,24)]
val=valid[valid.split=='validation']
means=val.groupby('surprise_group')[cols].mean().reindex(SURPRISE_GROUPS)
means.columns=['1 h','6 h','24 h']
ax=means.plot.bar(rot=0,color=['#31688e','#69a6a0','#d17a38'])
for i,group in enumerate(means.index):
    if means.loc[group].isna().all():
        ax.text(i,.5,'Sin datos',transform=ax.get_xaxis_transform(),ha='center',color='#555555')
ax.axhline(0,color='gray',lw=1); ax.set(title=f'Reacción media en validación — {DATA_MODE}',
    xlabel='Sorpresa de IPC respecto al consenso',ylabel='Retorno medio (bps)')
plt.tight_layout(); plt.show()
display(valid.groupby('split')[cols].count())

### Interpretación y límites
Busca si el signo de la reacción es consistente en desarrollo y validación, sin modificar el umbral después de observar la validación. Los datos sintéticos contienen un efecto programado: detectarlo verifica el método, no el mercado. Las medias de grupos pequeños son inestables; no se afirma significación estadística. Un estudio real necesita consenso archivado, timestamps exactos, control de otras noticias, independencia, intervalos de confianza y un test final reservado. Para simular una estrategia hay que entrar después de conocer la publicación, usar precios ejecutables bid/ask y descontar costes P08.

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · El método detecta el efecto programado, sobre todo a 1 hora</b>

Umbral fijado antes de mirar: ±0,15 pp. Frontera: 12-09-2024. Los 48 eventos son elegibles (son mensuales: no se solapan ni cruzan la frontera).

| Periodo | Grupo | n | Media a 1 h | Mediana a 1 h |
|---|---|---|---|---|
| Desarrollo | Inferior | 6 | **+45,6 bps** | +41,3 bps |
| Desarrollo | En línea | 19 | −0,3 bps | −5,1 bps |
| Desarrollo | Superior | 6 | **−44,6 bps** | −49,0 bps |
| Validación | Inferior | 3 | **+39,3 bps** | +39,2 bps |
| Validación | En línea | 14 | +15,7 bps | +12,7 bps |
| Validación | Superior | **0** | — | — |

- **El signo se mantiene:** una inflación por debajo del consenso va seguida de subidas de BTC en los dos periodos (+45,6 y +39,3 bps). Es el efecto programado: −1,5 % por cada punto de sorpresa, aplicado en la hora de la publicación.
- **La validación no tiene ningún evento "Superior":** su mayor sorpresa fue +0,146 pp, justo por debajo del umbral. Con 3 y 14 eventos, las medias son muy inestables.
- Detectarlo **verifica el método** (alineación, purga, umbral fijo), no el mercado: el efecto lo pusimos nosotros.

</div>

## 7. Comprobaciones reproducibles

In [ ]:
# 🔎 FASE DE VALIDACIÓN: fórmulas, causalidad temporal, purga y ausencia de solapes
checks={
 'PV conocido':np.isclose(present_value([110],.10),100),
 'bono a la par':np.isclose(bond_price(1000,.04,.04,5),1000),
 'precio decreciente con tipos':(bonds.price_eur.diff().dropna()<0).all(),
 'DCF perpetuidad sin crecimiento':np.isclose(dcf([100],.10,0)['enterprise_value'],1000),
 'DCF mayor tasa menor valor':dcf(flows,.09,.02)['enterprise_value']<base['enterprise_value'],
 'Fisher exacto':np.isclose(real_return(.05,.03),1.05/1.03-1),
 'inflación sin histórico ausente':macro.inflation_yoy_pct.iloc[:12].isna().all(),
 'consenso preevento':(events.consensus_at<events.available_at).all(),
 'baseline no futura':(results.pre_timestamp.dropna()<results.loc[results.pre_timestamp.notna(),'available_at']).all(),
 'purga temporal':(valid.loc[valid.split=='development','available_at']+pd.Timedelta(hours=24)<cutoff).all(),
 'sin solapes':not valid.overlap_24h.any(),
}
for h in (1,6,24):
    stamp=results[f'post_timestamp_{h}h']; ok=stamp.notna()
    checks[f'horizonte {h}h']=(stamp[ok]<=results.loc[ok,'available_at']+pd.Timedelta(hours=h)).all() and (stamp[ok]>=results.loc[ok,'available_at']).all()
for k,v in checks.items(): assert v,k
print(f'{len(checks)} comprobaciones superadas')
if EXPORT:
    OUTPUT_DIR.mkdir(exist_ok=True,parents=True)
    results.assign(data_mode=DATA_MODE).to_csv(OUTPUT_DIR/'event_study.csv',index=False)
    summary.to_csv(OUTPUT_DIR/'event_summary.csv')
    if fred:
        for key,value in fred.items(): value.to_csv(OUTPUT_DIR/f'{key}.csv')

## 8. Ejercicios
1. **Valor temporal:** implementa `my_pv`; comprueba que 110 € a un año con tasa 10% valen 100 € hoy.
2. **Bonos:** compara vencimientos de 2, 5 y 10 años con cupón 4% ante una subida de rendimiento del 4% al 5%. ¿Cuál pierde más en porcentaje?
3. **DCF:** calcula el valor por acción con r=10% y g=2%. ¿Por qué cambia aunque el negocio tenga los mismos flujos?
4. **Inflación:** compara la aproximación nominal menos inflación con la fórmula exacta para nominal 15% e inflación 10%.
5. **Curva de tipos:** localiza meses sintéticos con curva invertida y calcula su proporción. No interpretes inversión como predicción automática de recesión.
6. **Sorpresas:** calcula el retorno medio de los grupos a 1, 6 y 24 horas solamente en desarrollo. Contrasta después con la validación sin elegir el mejor horizonte a posteriori.
7. **Datos reales:** activa FRED, grafica inflación interanual y pendiente de curva, anota unidades, fechas y fuente. Explica por qué no puedes unir el IPC por mes de referencia en un backtest.
8. **Integración:** consigue una muestra de publicaciones con consenso anterior al evento y precios BTC/USD con timestamps fiables. Importa los CSV, cuantifica ausencias y redacta una hipótesis falsable para P11. No inventes consenso si no está disponible.

### E1 · Valor actual propio

In [ ]:
print(inspect.getsource(manual_pv))

my_pv = manual_pv
assert np.isclose(my_pv([110],.10),100)
assert np.isclose(my_pv(flows,.08), present_value(flows,.08))
print(f"110 € a un año al 10 %: {my_pv([110], .10):.2f} € | flujos del DCF al 8 %: {my_pv(flows, .08):.2f} M€ (igual que present_value)")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Valor actual</b>

- 110 / 1,10 = **100 €**: con una tasa del 10 %, recibir 110 € dentro de un año equivale a tener 100 € hoy.
- La implementación con bucle da lo mismo que `present_value` con los cinco flujos del DCF (429,92 M€).

</div>

### E2 · Bonos de 2, 5 y 10 años ante una subida del 4 % al 5 %

In [ ]:
print(inspect.getsource(bond_shock))

e2 = bond_shock((2, 5, 10), .04, .04, .05)
display(e2.round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · El bono más largo pierde más</b>

| Vencimiento | Precio al 4 % | Precio al 5 % | Variación |
|---|---|---|---|
| 2 años | 1.000,00 € | 981,41 € | **−1,86 %** |
| 5 años | 1.000,00 € | 956,71 € | **−4,33 %** |
| 10 años | 1.000,00 € | 922,78 € | **−7,72 %** |

- **Pierde más el de 10 años.** Sus flujos están más lejos en el tiempo, y cada punto de rendimiento se descuenta más veces: el factor (1 + r)^t pesa más cuanto mayor es t.
- Es la idea de **duración**: la sensibilidad del precio a los tipos crece con el vencimiento. Un bono "seguro" sin riesgo de impago puede perder casi un 8 % por una subida de tipos de un punto.

</div>

### E3 · DCF con r = 10 % y g = 2 %

In [ ]:
e3 = pd.DataFrame({'r = 8 %': dcf(flows, .08, .02), 'r = 10 %': dcf(flows, .10, .02)}).T
e3['value_per_share'] = [value_per_share(flows, r, .02, net_debt, shares) for r in (.08, .10)]
display(e3.round(2))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Mismo negocio, distinta exigencia</b>

| | r = 8 % | r = 10 % |
|---|---|---|
| Valor de los flujos explícitos | 429,92 M€ | 407,59 M€ |
| Valor terminal | 1.353,52 M€ | **926,15 M€** |
| Valor de empresa | 1.783,43 M€ | 1.333,74 M€ |
| Valor por acción | 14,83 € | **10,34 €** |

- Con r = 10 %, el valor por acción cae un **30 %** (de 14,83 a 10,34 €) **sin que cambie un solo flujo**.
- Cambia porque la tasa de descuento mide cuánto se **exige** por el riesgo y por esperar: con una exigencia mayor, cada euro futuro vale menos hoy.
- Casi toda la caída viene del **valor terminal** (−427 M€), porque su denominador es (r − g): de 6 a 8 puntos, un 33 % más. Los flujos explícitos apenas bajan (−22 M€).
- Por eso un DCF dice tanto de los supuestos de quien lo hace como del negocio.

</div>

### E4 · Fisher exacto frente a la aproximación

In [ ]:
print(inspect.getsource(fisher_comparison))

e4 = pd.DataFrame({'nominal 15 %, inflación 10 %': fisher_comparison(.15, .10),
                   'nominal 5 %, inflación 3 %': fisher_comparison(.05, .03)}).T
display(e4.style.format({'approx': '{:.3%}', 'exact': '{:.3%}', 'error_pp': '{:.3f}'}))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · La aproximación falla cuando la inflación es alta</b>

| | Aproximación (nominal − inflación) | Exacto (1 + n) / (1 + π) − 1 | Error |
|---|---|---|---|
| Nominal 15 %, inflación 10 % | 5,000 % | **4,545 %** | 0,45 pp |
| Nominal 5 %, inflación 3 % | 2,000 % | 1,942 % | 0,06 pp |

- La aproximación ignora el término cruzado: 1,15 / 1,10 = 1,0455, no 1,05.
- Con tipos bajos el error es pequeño (0,06 pp), pero con una inflación del 10 % la aproximación **sobrestima la rentabilidad real en casi medio punto**, un 10 % del resultado. En economías con inflación alta hay que usar siempre la fórmula exacta.

</div>

### E5 · Curva invertida en la serie sintética

In [ ]:
print(inspect.getsource(inverted_curve))

e5 = inverted_curve(macro)
print(f"Meses con curva invertida: {e5['count']} de {len(macro)} ({e5['share']:.1%}) | más invertida: {e5['deepest_bps']:.1f} bps")
display(macro.loc[macro.yield_curve_bps < 0, ['yield_2y_pct', 'yield_10y_pct', 'yield_curve_bps']])

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · 4 meses de 48 con la curva invertida</b>

- La curva sintética está invertida en **4 de 48 meses (8,3 %)**: de enero a abril de 2023, con un mínimo de **−1,3 bps**. Es una inversión mínima: el bono a 10 años rinde apenas una centésima menos que el de 2.
- La inversión aparece porque los rendimientos sintéticos son sinusoides con periodos distintos (`sin(t/10)` y `sin(t/13)`: unos 63 y 82 meses) y en esos meses el de 2 años alcanza al de 10. **No hay ninguna economía detrás.**
- En datos reales, una curva invertida suele reflejar que el mercado espera bajadas de tipos en el futuro, a menudo por una desaceleración. Pero no es una regla automática: el momento, la profundidad y la duración importan, y ha habido inversiones sin recesión posterior. No se puede usar como señal de recesión sin más evidencia.

</div>

### E6 · Horizontes de 1, 6 y 24 horas, primero solo en desarrollo

In [ ]:
print(inspect.getsource(development_horizons))

e6 = development_horizons(valid, (1, 6, 24))
print("Desarrollo:"); display(e6.loc['development'].round(2))
print("Validación (se mira después, con el mismo umbral y sin elegir horizonte):"); display(e6.loc['validation'].round(2))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · El efecto se ve mejor a 1 hora y se diluye con el horizonte</b>

Retorno medio (bps) por grupo; entre paréntesis, número de eventos:

| Grupo | Desarrollo 1 h | 6 h | 24 h | Validación 1 h | 6 h | 24 h |
|---|---|---|---|---|---|---|
| Inferior | +45,6 (6) | +42,8 | +74,9 | +39,3 (3) | +26,4 | +24,9 |
| En línea | −0,3 (19) | −4,1 | +3,2 | +15,7 (14) | +21,2 | +29,2 |
| Superior | −44,6 (6) | −51,5 | −43,4 | — (0) | — | — |

- **En desarrollo, el signo es coherente en los tres horizontes:** "Inferior" sube y "Superior" baja.
- **La separación es más limpia a 1 hora.** La correlación entre sorpresa y retorno es de −0,81 a 1 h y de −0,39 a 24 h. El efecto solo ocurre en la hora de la publicación, mientras que el ruido de las horas siguientes (unos 18 bps por hora) se acumula y lo tapa.
- **En validación**, "Inferior" mantiene el signo, pero "En línea" también sube (+16 a +29 bps): es ruido de un grupo de 14 eventos. No hay "Superior" que contrastar.
- No se eligió el horizonte mirando la validación. Si se hubiera elegido el de 24 h por su +74,9 bps en desarrollo, la validación lo habría desmentido (+24,9 bps).

</div>

### E7 · Datos reales de FRED
Cambia `RUN_FRED_E7` a `True` para descargar las seis series. Si la descarga falla, la celda se detiene con el error.

In [ ]:
print(inspect.getsource(fred_indicators))

RUN_FRED_E7 = False

if RUN_FRED_E7 or RUN_FRED:
    fred_e7 = fred if fred else {s: fetch_fred(s, FRED_START, FRED_END) for s in FRED_SERIES}
    print('Descarga UTC:', pd.Timestamp.now(tz='UTC'))
    display(pd.DataFrame({'n': {k: len(v) for k, v in fred_e7.items()},
                          'primero': {k: v.index.min().date() for k, v in fred_e7.items()},
                          'último': {k: v.index.max().date() for k, v in fred_e7.items()}}))
    monthly_real, curve_real = fred_indicators(fred_e7)
    print("Meses sin IPC publicado en FRED:", [d.date() for d in monthly_real.index[monthly_real.cpi_index.isna()]])
    display(monthly_real.tail(5).round(3))
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    monthly_real.inflation_yoy_pct.plot(ax=axes[0], marker='.', color='#31688e',
                                        title='FRED · IPC interanual (CPIAUCSL, %)', xlabel='Mes de referencia')
    curve_real.plot(ax=axes[1], color='#d17a38', title='FRED · DGS10 − DGS2 (bps, diario)', xlabel='Fecha')
    axes[1].axhline(0, color='gray', lw=1)
    plt.tight_layout(); plt.show()
    print(f"Curva invertida en {(curve_real < 0).mean():.1%} de los días; mínimo {curve_real.min():.0f} bps el {curve_real.idxmin().date()}")
else:
    print("RUN_FRED_E7 = False: no se consulta FRED. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · FRED (descarga del 19-11-2026, de 2022 a 2025)</b>

| Serie | Qué es | Unidad | Frecuencia | Observaciones |
|---|---|---|---|---|
| CPIAUCSL | IPC urbano de EE. UU., desestacionalizado | índice (1982–84 = 100) | mensual (mes de referencia) | **47 de 48** |
| FEDFUNDS | Tipo efectivo de fondos federales | % anual | mensual | 48 |
| DGS2 / DGS10 | Rendimiento del Tesoro a 2 y 10 años | % anual | diaria (días hábiles) | 998 |
| UNRATE | Tasa de paro | % | mensual | **47 de 48** |
| CBBTCUSD | BTC/USD en Coinbase | USD | diaria | 1.461 |

**Inflación interanual:** primer dato calculable en enero de 2023 (hacen falta 12 meses previos). Máximo **6,33 %** (enero de 2023), mínimo **2,33 %** (abril de 2025), **2,65 %** en diciembre de 2025.

**Curva 10 − 2 años:** invertida en el **54,2 %** de los días; mínimo **−108 bps** (3 de julio de 2023); dejó de estar invertida tras el 5 de septiembre de 2024 y terminó 2025 en +71 bps.

**Un hueco real:** FRED no tiene IPC ni paro de **octubre de 2025**. Si se calcula la inflación con `pct_change(12)` por posición, noviembre de 2025 sale **2,99 %**, porque se compara con octubre de 2024. Comparando por fecha, sale **2,70 %**. `yoy_by_date` deja octubre como hueco y compara cada mes con el mismo mes del año anterior.

**¿Por qué no se puede unir el IPC por mes de referencia en un backtest?** Porque el IPC de enero **no se conoce en enero**: se publica a mediados de febrero, a una hora concreta. Unirlo a los precios de enero metería en el pasado un dato que nadie tenía (*look-ahead*). Además, esta descarga es la versión **revisada de hoy**: los valores publicados en su momento pueden ser distintos. Hace falta la fecha y hora real de publicación (`available_at`) y el valor de esa primera publicación (vintage de ALFRED).

</div>

### E8 · Integración: ¿hay datos suficientes para un estudio real?

In [ ]:
print(inspect.getsource(availability_report))

# Lo que se puede construir solo con FRED: un evento por mes de referencia, con el dato revisado de hoy.
# Inflación interanual de CPIAUCSL: se calcula con fred_indicators si E7 descargó FRED; si no, se usan los
# valores transcritos de la salida de E7 (FRED, descargado el 19-11-2026).
if 'monthly_real' in globals():
    actual = monthly_real.inflation_yoy_pct.loc['2023-01':'2023-03'].round(2).tolist()
else:
    actual = [6.33, 5.96, 4.92]  # transcritos de E7, no recalculados sin conexión
# FRED no trae la hora de publicación ni el consenso: se dejan vacíos en lugar de inventarlos.
fred_only = pd.DataFrame({'event_id': ['CPI-2023-01', 'CPI-2023-02', 'CPI-2023-03'],
                          'reference_month': ['2023-01', '2023-02', '2023-03'],
                          'available_at': pd.NaT, 'consensus_at': pd.NaT,
                          'actual_pct': actual, 'consensus_pct': np.nan})
display(fred_only)
display(availability_report(fred_only))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Sin consenso no hay estudio: evidencia insuficiente</b>

**Lo que hay disponible gratis:**
- El **dato** del IPC (FRED), pero revisado: no necesariamente el que se publicó.
- Los **precios diarios** de BTC/USD (CBBTCUSD), insuficientes para medir una reacción a 1 hora.

**Lo que falta**, según el informe de ausencias:
- El **consenso** previo a cada publicación: **100 % ausente**. Sin él no hay sorpresa que medir. Las encuestas de consenso (Bloomberg, Reuters, etc.) son de pago, y no se puede reconstruir a posteriori.
- La **hora exacta de publicación** (`available_at`): 100 % ausente en FRED. Habría que sacarla del calendario oficial de publicaciones del BLS.
- **Precios intradía** de BTC/USD con timestamps fiables alrededor de las 13:30 UTC (por ejemplo, velas de 1 minuto del exchange, usando el fin de cada vela).

Con estos datos, el estudio real **no se puede hacer**: no se inventa el consenso, y la conclusión es **evidencia insuficiente**.

**Hipótesis falsable para P11.** *En las publicaciones del IPC de EE. UU. entre 2022 y 2025, la sorpresa (dato − consenso archivado antes de la publicación) tiene una correlación negativa con el retorno de BTC/USD en Coinbase desde el último precio antes de las 13:30 UTC hasta 60 minutos después.*
- **Datos:** consenso de una fuente archivada con su fecha, hora oficial del BLS y velas de 1 minuto de Coinbase.
- **Criterio de rechazo:** se rechaza si, en el último año reservado (nunca usado para fijar nada), la correlación es ≥ 0, o si un test de permutación (barajar las sorpresas entre eventos, como hará P11) no la distingue del azar al 5 %.
- **Control:** el mismo cálculo en días sin publicación a la misma hora, para descartar un patrón horario que no tenga nada que ver con el IPC.

</div>

## 9. Conclusiones

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P10</b>

**¿Qué diferencia existe entre precio, valor y retorno real?**
El **precio** es lo que se paga hoy en el mercado (15 € por acción). El **valor** es una estimación de lo que valen los flujos futuros, y depende de supuestos: con los mismos flujos va de 6,40 a 47,00 € según r y g. El **retorno real** es la ganancia descontada la inflación: con un 15 % nominal y un 10 % de inflación, es un 4,55 %, no un 5 %.

**¿Qué variables estaban disponibles antes de decidir y cuáles son etiquetas futuras?**
Antes de la publicación estaban disponibles el **consenso** (registrado el día anterior) y el **precio previo** (estrictamente anterior a la hora de publicación). El **dato publicado** y la **sorpresa** solo existen desde `available_at`. Los **retornos a 1, 6 y 24 horas** son etiquetas: se usan para evaluar, nunca como features.

**¿Qué patrón sobrevive a la validación? ¿Cuántos eventos lo sostienen?**
Solo el del grupo "Inferior": una inflación por debajo del consenso va seguida de subidas de BTC (+45,6 bps a 1 h en desarrollo, +39,3 bps en validación). Lo sostienen **6 + 3 eventos**. El grupo "Superior" no se pudo validar (0 eventos), y el efecto es el que programamos. No hay evidencia sobre el mercado real.

**¿Qué fuente adicional necesitarías antes de utilizar esta señal?**
Un **consenso archivado** antes de cada publicación, la **hora oficial** de publicación del BLS, el **primer valor publicado** (vintage de ALFRED, no la serie revisada), **precios intradía ejecutables** (bid/ask) de BTC con timestamps fiables, un registro de **otras noticias** simultáneas y los **costes de P08**. Sin consenso, la conclusión honesta es evidencia insuficiente.

📄 Resumen en `docs/conclusiones/P10_fundamental_macro.md`

</div>

**Siguiente práctica: P11 — eficiencia del mercado, random walk y contraste de señales frente al azar.**